In [7]:
import os, json, time, random
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_curve, auc

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle


In [8]:
# =========================
# User config
# =========================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --- dataset loading (user-provided) ---
dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

# from <your_loader_module> import load_compact_pkl_dataset
compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)

# --- subset sizes ---
KNOWN_TX_NUM = 4             # 4 known + 2 unknown (for 6-TX setup)
SOURCE_RX_NUM = 3            # randomly choose X source RXs using SEED

# --- training / test dates ---
TRAIN_DATES = ["2021_03_15"]       # source day(s)
TEST_DATES_RX = ["2021_03_15"]     # for cross-RX drift (same day)
TEST_DATES_DAY = ["2021_03_01"]    # for cross-day drift (same RX)


# --- which signal version to use ---
Z_FROM_EQ = 1               # classifier input: 1=equalized, 0=unequalized
D_FROM = "raw"              # domain descriptor: "raw" or "eq" (recommended: "raw")

MAX_SIG_PER_TRIPLE = None   # per (tx,rx,day,eq) max signals; None=use all

# --- training hyperparams ---
BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8

IN_PLANES = 64
DROPOUT = 0.3

# --- domain descriptor dim ---
D_DIM = 32

# --- output dir ---
ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"./results_wisig_module2_stratified/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED, dataset_name=dataset_name, dataset_path=dataset_path,
    KNOWN_TX_NUM=KNOWN_TX_NUM, SOURCE_RX_NUM=SOURCE_RX_NUM,
    TRAIN_DATES=TRAIN_DATES, TEST_DATES_RX=TEST_DATES_RX, TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ, D_FROM=D_FROM,
    MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES, DROPOUT=DROPOUT, D_DIM=D_DIM
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)
print("RUN_DIR:", RUN_DIR)


Device: cuda
RUN_DIR: ./results_wisig_module2_stratified/run_20260304_141838


In [9]:
def get_idx(lst, val):
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    # Return np.ndarray shape (N,256,2), dtype float32
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x_256_2):
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    # generic FFT log-magnitude low-frequency coefficients
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X_256_2, d_dim=32):
    Xc = iq_to_complex(X_256_2)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)


In [10]:
# Choose 6 TX, 12 RX, 4 DAY and randomly choose SOURCE_RX_NUM source RXs
tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:6]
RX_USE = rx_list[:12]
DATE_USE = date_list[:4]

KNOWN_TX = TX_USE[:KNOWN_TX_NUM]
UNKNOWN_TX = TX_USE[KNOWN_TX_NUM:]

rng = np.random.RandomState(SEED)
SOURCE_RXS = list(rng.choice(RX_USE, size=SOURCE_RX_NUM, replace=False))
SOURCE_RXS.sort()
DRIFT_RX = [r for r in RX_USE if r not in SOURCE_RXS]

print("KNOWN_TX:", KNOWN_TX)
print("UNKNOWN_TX:", UNKNOWN_TX)
print("SOURCE_RXS:", SOURCE_RXS)
print("DRIFT_RX:", DRIFT_RX)
print("TRAIN_DATES:", TRAIN_DATES)
print("TEST_DATES_RX:", TEST_DATES_RX)
print("TEST_DATES_DAY:", TEST_DATES_DAY)


KNOWN_TX: ['14-10', '14-7', '20-15', '20-19']
UNKNOWN_TX: ['6-15', '8-20']
SOURCE_RXS: ['1-1', '7-14', '7-7']
DRIFT_RX: ['1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '8-8']
TRAIN_DATES: ['2021_03_15']
TEST_DATES_RX: ['2021_03_15']
TEST_DATES_DAY: ['2021_03_01']


In [11]:
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)  # (B,256,2)->(B,2,256)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)  # (B,512)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits


In [12]:
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        pred = logits.argmax(dim=1)
        total_correct += int((pred == yb).sum().item())
        total += int(yb.size(0))
    return total_loss / max(1,total), total_correct / max(1,total)

def compute_embeddings(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    Z = []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device)
            _, emb = model(xb, return_embed=True)
            Z.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(Z, axis=0)

def l2norm(x, axis=1, eps=1e-12):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

def prototypes(Z, y, K):
    ZN = l2norm(Z, axis=1)
    P = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        idx = np.where(y==k)[0]
        P[k] = ZN[idx].mean(axis=0)
    return l2norm(P, axis=1)

def cosine_to_proto(Z, P):
    ZN = l2norm(Z, axis=1)
    return ZN @ P.T

def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + 1e-3*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    return mu, Sinv

def sdom_min_to_sources(D, source_models):
    dists = []
    for (mu, Sinv) in source_models:
        dists.append(mahalanobis_batch(D, mu, Sinv))
    return np.min(np.stack(dists, axis=1), axis=1).astype(np.float32)


In [13]:
def build_source_train(compact_dataset):
    X_list, y_list, D_list = [], [], []
    rx_id_list, day_id_list = [], []

    for tx in KNOWN_TX:
        for rx in SOURCE_RXS:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                if D_FROM == "raw":
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0), MAX_SIG_PER_TRIPLE)
                else:
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 1), MAX_SIG_PER_TRIPLE)

                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]
                Xd = Xd[:n]

                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)

                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
                day_id_list.append(np.full((n,), TRAIN_DATES.index(day), dtype=np.int64))

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    y = np.concatenate(y_list, axis=0).astype(np.int64)
    D = np.concatenate(D_list, axis=0).astype(np.float32)
    rx_id = np.concatenate(rx_id_list, axis=0).astype(np.int64)
    day_id = np.concatenate(day_id_list, axis=0).astype(np.int64)
    return X, y, D, rx_id, day_id

def build_eval_set(compact_dataset, txs, rxs, days):
    X_list, y_list, D_list = [], [], []

    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                if D_FROM == "raw":
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0), MAX_SIG_PER_TRIPLE)
                else:
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 1), MAX_SIG_PER_TRIPLE)

                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]
                Xd = Xd[:n]

                if tx in KNOWN_TX:
                    y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                else:
                    y = np.full((n,), -1, dtype=np.int64)

                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)

                X_list.append(Xz); y_list.append(y); D_list.append(Df)

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    y = np.concatenate(y_list, axis=0).astype(np.int64)
    D = np.concatenate(D_list, axis=0).astype(np.float32)
    return X, y, D

X_src, y_src, D_src, rx_src, day_src = build_source_train(compact_dataset)
print("Source:", X_src.shape, y_src.shape, D_src.shape)


Source: (12000, 256, 2) (12000,) (12000, 32)


In [14]:
# Evaluation scenarios
X_drift_rx, y_drift_rx, D_drift_rx = build_eval_set(compact_dataset, KNOWN_TX, DRIFT_RX, TEST_DATES_RX)
X_unk_in, y_unk_in, D_unk_in = build_eval_set(compact_dataset, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_RX)
X_drift_day, y_drift_day, D_drift_day = build_eval_set(compact_dataset, KNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)

print("Known drift RX:", X_drift_rx.shape)
print("Unknown in-domain:", X_unk_in.shape)
print("Known drift DAY:", X_drift_day.shape)


Known drift RX: (36000, 256, 2)
Unknown in-domain: (6000, 256, 2)
Known drift DAY: (12000, 256, 2)


In [15]:
def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def plot_curves(hist, out_png):
    e = np.arange(1, len(hist["train_loss"])+1)
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(e, hist["train_loss"], label="train")
    plt.plot(e, hist["val_loss"], label="val")
    plt.grid(True); plt.legend(); plt.xlabel("epoch"); plt.ylabel("loss")
    plt.subplot(1,2,2)
    plt.plot(e, hist["train_acc"], label="train")
    plt.plot(e, hist["val_acc"], label="val")
    plt.grid(True); plt.legend(); plt.xlabel("epoch"); plt.ylabel("acc")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def plot_confmat(cm, out_png, title):
    plt.figure(figsize=(5,4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("pred"); plt.ylabel("true")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def plot_hist(a, b, la, lb, title, out_png, bins=80):
    plt.figure(figsize=(6,4))
    plt.hist(a, bins=bins, density=True, alpha=0.6, label=la)
    plt.hist(b, bins=bins, density=True, alpha=0.6, label=lb)
    plt.grid(True); plt.legend()
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def plot_roc(y_true, score, title, out_png):
    fpr, tpr, _ = roc_curve(y_true, score)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],"--")
    plt.grid(True); plt.legend()
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()
    return float(roc_auc)


In [16]:
K = len(KNOWN_TX)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_summ = []

for fold, (idx_tr, idx_te) in enumerate(skf.split(X_src, y_src), start=1):
    fold_dir = os.path.join(RUN_DIR, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)

    # val split from train indices
    rng = np.random.RandomState(SEED + fold)
    perm = rng.permutation(idx_tr)
    n_val = max(1, int(0.1 * len(perm)))
    idx_val = perm[:n_val]
    idx_train = perm[n_val:]

    X_train, y_train = X_src[idx_train], y_src[idx_train]
    X_val, y_val = X_src[idx_val], y_src[idx_val]
    X_test, y_test = X_src[idx_te], y_src[idx_te]  # stable in-domain

    train_loader = DataLoader(ArrayDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(ArrayDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(ArrayDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

    model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val = -1.0
    best_state = None
    patience = 0
    hist = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

    for ep in range(1, EPOCHS+1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer=opt)
        va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)

        hist["train_loss"].append(tr_loss)
        hist["train_acc"].append(tr_acc)
        hist["val_loss"].append(va_loss)
        hist["val_acc"].append(va_acc)

        if va_acc > best_val + 1e-4:
            best_val = va_acc
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                break

    torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))
    save_json(os.path.join(fold_dir, "history.json"), hist)
    plot_curves(hist, os.path.join(fold_dir, "train_curves.png"))

    model.load_state_dict(best_state)

    # in-domain test acc
    model.eval()
    y_pred = []
    with torch.no_grad():
        for xb, _yb in test_loader:
            xb = xb.to(device)
            logits = model(xb)
            y_pred.append(logits.argmax(dim=1).cpu().numpy())
    y_pred = np.concatenate(y_pred)
    acc = float((y_pred == y_test).mean())
    cm = confusion_matrix(y_test, y_pred, labels=list(range(K)))
    plot_confmat(cm, os.path.join(fold_dir, "confmat_in_domain.png"), "In-domain Confusion")

    # prototypes for S_id
    Z_train = compute_embeddings(model, X_src[idx_train], batch=512)
    P = prototypes(Z_train, y_src[idx_train], K)

    def sid_from_Z(Z):
        cos = cosine_to_proto(Z, P)
        return np.max(cos, axis=1).astype(np.float32)

    # per-RX gaussian models (domain) from TRAIN split (all TX pooled)
    source_models_rx = []
    for rx in SOURCE_RXS:
        rx_idx = RX_USE.index(rx)
        sel = idx_train[np.where(rx_src[idx_train] == rx_idx)[0]]
        mu, Sinv = fit_gaussian(D_src[sel])
        source_models_rx.append((mu, Sinv))

    # per-day gaussian models from TRAIN split (all TX pooled)
    source_models_day = []
    for di, day in enumerate(TRAIN_DATES):
        sel = idx_train[np.where(day_src[idx_train] == di)[0]]
        mu, Sinv = fit_gaussian(D_src[sel])
        source_models_day.append((mu, Sinv))

    # Scores: A stable
    Z_A = compute_embeddings(model, X_test, batch=512)
    Sid_A = sid_from_Z(Z_A)
    Sdom_rx_A = sdom_min_to_sources(D_src[idx_te], source_models_rx)
    Sdom_day_A = sdom_min_to_sources(D_src[idx_te], source_models_day)

    # Scores: B drift RX
    Z_B = compute_embeddings(model, X_drift_rx, batch=512)
    Sid_B = sid_from_Z(Z_B)
    Sdom_rx_B = sdom_min_to_sources(D_drift_rx, source_models_rx)

    # Scores: C unknown in-domain
    Z_C = compute_embeddings(model, X_unk_in, batch=512)
    Sid_C = sid_from_Z(Z_C)

    # Scores: E drift day
    Z_E = compute_embeddings(model, X_drift_day, batch=512)
    Sid_E = sid_from_Z(Z_E)
    Sdom_day_E = sdom_min_to_sources(D_drift_day, source_models_day)

    np.savez_compressed(os.path.join(fold_dir, "scores_minimal.npz"),
                        Sid_A=Sid_A, Sdom_rx_A=Sdom_rx_A, Sdom_day_A=Sdom_day_A,
                        Sid_B=Sid_B, Sdom_rx_B=Sdom_rx_B,
                        Sid_C=Sid_C,
                        Sid_E=Sid_E, Sdom_day_E=Sdom_day_E)

    # AUC unknown: stable vs unknown-in (score=-Sid)
    y_u = np.concatenate([np.zeros_like(Sid_A, dtype=np.int64), np.ones_like(Sid_C, dtype=np.int64)])
    s_u = np.concatenate([-Sid_A, -Sid_C])
    auc_u = plot_roc(y_u, s_u, "Unknown ROC (score=-S_id)", os.path.join(fold_dir, "roc_unknown.png"))

    # AUC drift RX: stable vs known drift RX (score=Sdom_rx)
    y_dr = np.concatenate([np.zeros_like(Sdom_rx_A, dtype=np.int64), np.ones_like(Sdom_rx_B, dtype=np.int64)])
    s_dr = np.concatenate([Sdom_rx_A, Sdom_rx_B])
    auc_drRX = plot_roc(y_dr, s_dr, "Drift ROC cross-RX (score=S_dom_rx)", os.path.join(fold_dir, "roc_drift_rx.png"))

    # AUC drift day: stable vs drift day (score=Sdom_day)
    y_dd = np.concatenate([np.zeros_like(Sdom_day_A, dtype=np.int64), np.ones_like(Sdom_day_E, dtype=np.int64)])
    s_dd = np.concatenate([Sdom_day_A, Sdom_day_E])
    auc_drDAY = plot_roc(y_dd, s_dd, "Drift ROC cross-day (score=S_dom_day)", os.path.join(fold_dir, "roc_drift_day.png"))

    # Histograms
    plot_hist(Sid_A, Sid_C, "Known(stable)", "Unknown(in-domain)", "S_id: Known vs Unknown",
              os.path.join(fold_dir, "hist_Sid_known_vs_unknown.png"))
    plot_hist(Sdom_rx_A, Sdom_rx_B, "Stable", "Known-Drift(RX)", "S_dom_rx: Stable vs Drift(RX)",
              os.path.join(fold_dir, "hist_Sdom_rx_stable_vs_drift.png"))
    plot_hist(Sdom_day_A, Sdom_day_E, "Stable", "Known-Drift(DAY)", "S_dom_day: Stable vs Drift(DAY)",
              os.path.join(fold_dir, "hist_Sdom_day_stable_vs_drift.png"))

    metrics = dict(fold=fold, acc=acc, AUC_u=auc_u, AUC_drRX=auc_drRX, AUC_drDAY=auc_drDAY)
    save_json(os.path.join(fold_dir, "metrics.json"), metrics)
    fold_summ.append(metrics)

    print(f"[Fold {fold}] acc={acc:.3f}  AUC_u={auc_u:.3f}  AUC_drRX={auc_drRX:.3f}  AUC_drDAY={auc_drDAY:.3f}")

save_json(os.path.join(RUN_DIR, "metrics_all_folds.json"), fold_summ)
print("Mean acc:", np.mean([m["acc"] for m in fold_summ]))
print("Mean AUC unknown:", np.mean([m["AUC_u"] for m in fold_summ]))
print("Mean AUC drift_rx:", np.mean([m["AUC_drRX"] for m in fold_summ]))
print("Mean AUC drift_day:", np.mean([m["AUC_drDAY"] for m in fold_summ]))


[Fold 1] acc=0.999  AUC_u=0.945  AUC_drRX=0.681  AUC_drDAY=0.578
[Fold 2] acc=0.996  AUC_u=0.938  AUC_drRX=0.681  AUC_drDAY=0.581
[Fold 3] acc=0.993  AUC_u=0.896  AUC_drRX=0.699  AUC_drDAY=0.586
[Fold 4] acc=0.997  AUC_u=0.937  AUC_drRX=0.681  AUC_drDAY=0.588
[Fold 5] acc=0.994  AUC_u=0.956  AUC_drRX=0.685  AUC_drDAY=0.583
Mean acc: 0.9957499999999999
Mean AUC unknown: 0.9344503749999999
Mean AUC drift_rx: 0.6853392118055556
Mean AUC drift_day: 0.583033638888889
